# Comparison program

Canonical source for `BENCHMARK.md`. Export the complete fresh comparison runtime from the repository root:

```sh
python bench/comparison_loader.py NEW_DIRECTORY
```

Use the original command documented below with the exported program path. External runtimes and native defaults remain those specified by the program. This notebook migration changes source ownership, not model parameters or recorded predictions.


# vdjmatch comparison benchmark

Plan + protocol for the cross-method epitope-annotation benchmark (manuscript Deliverable 3). The
companion `EXTERNAL_TOOLS.md` covers sandboxing the non-`vdjmatch` methods; the CLI is `bench/compare.py`.

## Goal

For each method, report **per-locus (TRA / TRB), per-epitope** annotation quality as a *distribution*
over epitopes — **F1, PR-AUC, retention, purity** — drawn as **boxplots + beeswarm** across methods, on
the latest VDJdb release and its high-confidence shortlist, plus four held-out samples. We are
deliberately strict with ourselves: `vdjmatch` may **never** use an exact CDR3 match to a benchmark
query (those reference rows are removed; see *Leakage*).

## Datasets

Reference DB = **VDJdb 2026-06-11-ZENODO** (`db.fetch_hf`), with its **≥2-reference shortlist**
(`db.replicated`) as the gold standard. Query sets (in `2026-vdjmatch/test_data/`):

| set | file | chain(s) | role | truth |
|---|---|---|---|---|
| full VDJdb | (HF release) | TRA, TRB | leave-one-out-by-epitope retrieval | epitope label |
| shortlist | (HF, ≥2 ref) | TRA, TRB | gold-standard LOO | epitope label |
| sample1 | `sample1_airr.txt` (489) | TRB | NLVPMVATV (CMV) sorted+ | **positives** (all NLV) |
| sample2 | `sample2_airr.txt` (319) | TRB | LLWNGPMAV (YF) + LLL… (BST-2 neoag) sorted+ | **positives** |
| sample3 | `sample3_airr.txt` / `sample3_vdjdb.txt` | TRA,TRB | VDJdb 2025 (older release) | epitope label |
| sample5 | `sample5_olga_airr.txt` (300k) | TRB | random OLGA repertoire | **negatives** (no hits expected) |
| sample6 | `sample6_TCRvdb.csv` (3693, paired) | TRA+TRB | TCRvdb, 2 epitopes | **pos/neg by `padj < 1e-5`** |

`sample6_vdb_airr.txt` is the single-chain (β) view of the same TCRvdb data; `sample6_TCRvdb.csv` is the
paired (α+β) table with `cdr3_alpha_aa`, `cdr3_beta_aa`, `epitope_aa`, `log2FoldChange`, `pvalue`,
`padj` — used for the **paired-chain E-value** test (positives = enriched binders at `padj < 1e-5`).

## Leakage removal (be fair to ourselves)

Before annotating any benchmark query with `vdjmatch`, drop from the reference DB **every row whose
CDR3 (per locus) exactly matches a query CDR3**, and for full-VDJdb / shortlist LOO also exclude the
held-out epitope's own clonotypes. So a correct call must come from *other* TCRs, never a database
copy of the query. (sample5 OLGA negatives need no removal — they shouldn't be in VDJdb at all; if any
are, that is itself a measured false-positive.)

## Methods (algorithm list — expand later)

Pluggable `Method` in `compare.py`; external ones read a standard predictions file produced in their
own conda sandbox (`EXTERNAL_TOOLS.md`).

- **vdjmatch** (ours): control-calibrated E-value over the leakage-removed reference; paired E-value for
  sample6.
- **TCRMatch** — CDR3β k-mer similarity to IEDB/VDJdb (PMID 33777034).
- **tcrdist3** — TCRdist nearest-neighbour over V+CDR3 (Mayer-Blackwell 2021).
- **GLIPH2** — motif/global clustering, then label transfer (Huang 2020).
- **ERGO-II** — supervised TCR–peptide binding (LSTM/AE; PMID 33981311).
- **ImmuneWatch DETECT** — commercial; accuracy-only from provided outputs (`immunedetect_results/`).
- *(placeholders for the user to add: NetTCR-2.2, pMTnet, STAPLER, TCRBert, MixTCRpred, …)*

A "method" maps (query CDR3[s], V/J, locus) → per-epitope score (higher = more likely binder), plus a
significance/decision flag where it has one.

## Metrics (per locus × per epitope)

For epitope *E* and locus *L*, restrict to queries of locus *L*; the binary task is "does this query
bind *E*". Let the method give each query a score for *E* and a binary call (its significant set).

- **PR-AUC** — area under precision–recall, ranking queries by the *E*-score (prevalence-aware;
  `metrics.pr_auc`; balanced variant for cross-method comparability where prevalence differs).
- **F1** — at the method's own decision threshold (E-value significant / score cutoff): `2PR/(P+R)`.
- **purity** — precision of the method's *E*-set: fraction of queries it calls *E* that truly bind *E*
  (set homogeneity).
- **retention** — recall after the significance filter: fraction of true-*E* queries that keep ≥1
  significant same-*E* hit. (Purity = "are the calls clean", retention = "did we keep the true ones".)

Only epitopes with ≥ N (default 30) labelled queries in that locus enter the distribution. sample5 has
no true epitope → scored as **specificity / FPR** (fraction of OLGA queries wrongly called significant).

## Plots

Per metric: one panel per locus, x = method, y = per-epitope value, **boxplot (quartiles) + beeswarm
(one dot per epitope)** overlaid (seaborn `boxplot` + `swarmplot`). Paired Wilcoxon vdjmatch-vs-each
across the shared epitopes for the stats line.

## Paired-chain (sample6 TCRvdb)

Run `vdjmatch` paired α+β E-values; positives = `padj < 1e-5`, negatives = the rest. Report ROC-AUC /
PR-AUC of the paired E-value as a binder classifier, and check paired E < min(single-chain E) for true
pairs. Single-chain β baseline from `sample6_vdb_airr.txt`.

## CLI

```fish
# vdjmatch arm on the annotated samples (leakage-removed reference)
python bench/compare.py --methods vdjmatch --datasets sample1 sample2 sample5 --out bench/out
# add external methods once their predictions/<method>/<dataset>.tsv files exist
python bench/compare.py --methods vdjmatch tcrmatch --datasets shortlist --out bench/out
```
Emits `bench/out/results.tsv` (tidy: method, dataset, locus, epitope, metric, value) + the boxplot/
beeswarm PNGs.

### Junction geometry arms

`bench/geometry_detect.py` is the hook for a **vdjmatch junction geometry**. Until it existed no
script here exercised one, and the geometry selected by the four-task objective was reproducible
only from the analysis workspace.

```fish
# requires the supplied samples; it tells you so if they are missing
set -x VDJMATCH_TESTDATA ~/vcs/manuscripts/2026-vdjmatch-ms/test_data

python bench/geometry_detect.py                       # shipped TCRdist3 geometry
python bench/geometry_detect.py --stem-depth 4        # the selected joint V-J stem arm
python bench/geometry_detect.py --compare             # both, with the per-task delta
```

It runs the Group-A detection tasks — NLV+/tet-, LLW vs LLL, and YLQ/GLC on TCRvdb, i.e. the
leakage-free stratum where every external method is comparable — and writes the standard
`predictions/<method>/<task>_<locus>.tsv`, so its output drops into `compare.py` beside
`tcrdist`, `imw` and the rest. Geometry knobs: `--stem-depth`, `--stem-weight`,
`--junction-ends`, `--v-loop-weight`, `--alphabet`.

**It scores nearest-neighbour distance and an in-radius kernel sum, matching
`tcrdist_detect.py`, not the kernel-ladder statistic.** That keeps it like-for-like with the
other producers and means its numbers are *not* the ladder-statistic numbers recorded in the
analysis workspace and must not be quoted as them. What it measures is the geometry, with the
scoring held fixed.

Measured 13 October 2026, `--compare`, 7 tasks over both loci, stem depth 4 against depth 0:
mean **+0.0032 ROC** and **+0.0059 PR**, improving 5 of 7. The gain concentrates on alpha —
GLC TRA +0.0085 ROC, YLQ TRA +0.0060 — and both small losses are TRB (YLQ TRB −0.0012, LLL TRB
−0.0002). That is the alpha-chain asymmetry the sweep predicted, recovered here under a
different scoring rule and a different task construction, so it is corroboration rather than a
repeat of the same measurement.

Exact junction identities are punctured from the reference. Queries whose V call does not
resolve are dropped and counted in the per-task line, never silently.

## Figures/tables selected for the appendix (now)

Keep the small set that carries the argument; cut the rest (text update later, from `docs/`):

- **Fig — purity vs distance** (Hamming-1 optimum) — keep.
- **Fig — matrices bar** (no AA matrix beats BLOSUM62; VDJAMr ties; possig wins) — keep.
- **Fig — position significance** (end-anchored profile) — keep.
- **Fig — vgene_scan** (V prior universal) — keep.
- **Table — matrices** (balanced PR-AUC) and **Table — vregion** (near-exact recovery) — keep.
- **This benchmark's boxplot+beeswarm** (cross-method, per-epitope) — the headline of Deliverable 3.
- Cut/condense: tra_trb_gly, region_corr, retention, loo_prauc, vgene/vgene-sim/vgene-scan tables,
  shortlist table → fold into one or two summary panels.

## Status (vdjmatch arm, framework)

- **Done:** shortlist/VDJdb LOO per-epitope metrics + boxplot/beeswarm (`--datasets shortlist`);
  **samples mode** (`--datasets samples`) — NLV / LLW / LLL ROC+PR (positives vs OLGA negatives) +
  the **OLGA spurious-hit filtering** panel. vdjmatch flags only **0.003%** of 240k random OLGA β
  queries (7 hits) at subs 1 — the noise-filtering headline.
- **Next (no external deps):** the two big sorted tables (shortlist; full VDJdb ≥30 ≤3000) with
  f1/precision/recall/retention/purity by species×chain×MHC×epitope; the paired TCRvdb ROC/PR
  (sample6, `padj < 1e-5`).
- **Next (external):** sandbox TCRMatch, tcrdist3 (1-NN+k-NN), GLIPH2 (osx binary → immGLIPH fallback),
  ERGO-II (pretrained), DETECT (`.ods`, score col `Score`, `predictions_sampleN` ↔ `sampleN`); add the
  easy-to-install extras; ROC/PR curves for **top-2 + vdjmatch** only. Decisions recorded in
  `EXTERNAL_TOOLS.md`.

## Open questions

Answered in `EXTERNAL_TOOLS.md` (Q1–Q6 + method/reporting plan).
